In [5]:
# # 必要なパッケージの読み込み
# library(caret)
# library(Metrics)
# library(ggplot2)
# library(lattice)

# # 対象ファイル名一覧
# file_names <- c(
#   "n_base.csv", "n_base_OH.csv", "n_base_FP.csv",
#   "n_all.csv", "n_all_OH.csv", "n_all_FP.csv",
#   "m_base.csv", "m_base_OH.csv", "m_base_FP.csv",
#   "m_all.csv", "m_all_OH.csv", "m_all_FP.csv"
# )

# # データパスの共通部分
# base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/"

# # モデルの出力用日付
# today <- format(Sys.Date(), "%Y%m%d")

# # 評価結果の格納用データフレームの初期化
# target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
# metric_names <- c("Best_num.labels", "Best_type.mf", "R2", "RMSE", "MAE", "RPD", "n_samples", "n_features")
# result_matrix <- matrix(nrow = length(metric_names) * length(target_vars), ncol = length(file_names))
# rownames(result_matrix) <- as.vector(t(outer(metric_names, target_vars, paste, sep = "_")))
# colnames(result_matrix) <- file_names


# #--- メインループ ---#
# for (file in file_names) {
#   filepath <- paste0(base_path, file)
#   cat("\n=== Processing:", file, "===\n")
#   df_all <- read.csv(filepath)
#   cat("Final dataset size:", dim(df_all)[1], dim(df_all)[2], "\n")

#   feature_vars <- setdiff(colnames(df_all), target_vars)

#   for (target_var in target_vars) {
#     cat("\n---\nTraining model for:", target_var, "on", file, "\n")
#     df <- df_all[, c(feature_vars, target_var)]
#     df <- df[complete.cases(df), ]

#     # モデルチューニング（仮定: alpha, lambda）
#     tune_grid <- expand.grid(
#       num.labels = c(3, 5, 7),
#       type.mf = c("GAUSS", "TRIANGLE")
#     )

#     ctrl <- trainControl(method = "cv", number = 10)

#     model <- train(
#       formula(paste(target_var, "~ .")),
#       data = df,
#       method = "WM",
#       metric = "RMSE",
#       trControl = ctrl,
#       tuneGrid = tune_grid
#     )

#     pred <- predict(model, df)
#     obs <- df[[target_var]]

#     R2 <- round(cor(pred, obs)^2, 3)
#     RMSE_val <- round(rmse(obs, pred), 3)
#     MAE_val <- round(mae(obs, pred), 3)
#     RPD_val <- round(sd(obs) / RMSE_val, 3)

#     cat("Best params: alpha =", model$bestTune$alpha,
#         ", lambda =", model$bestTune$lambda, "\n")
#     cat(file, target_var, ": R2 =", R2, ", RMSE =", RMSE_val, ", MAE =", MAE_val, ", RPD =", RPD_val, "\n")

#     # 結果保存
#     result_matrix[paste0("Best_num.labels_", target_var), file] <- model$bestTune$num.labels
#     result_matrix[paste0("Best_type.mf_", target_var), file]    <- model$bestTune$type.mf

#     result_matrix[paste0("R2_", target_var), file] <- R2
#     result_matrix[paste0("RMSE_", target_var), file] <- RMSE_val
#     result_matrix[paste0("MAE_", target_var), file] <- MAE_val
#     result_matrix[paste0("RPD_", target_var), file] <- RPD_val
#     result_matrix[paste0("n_samples_", target_var), file] <- nrow(df)
#     result_matrix[paste0("n_features_", target_var), file] <- ncol(df) - 1

#     # 変数重要度（varImp対応時のみ）
#     try({
#       importance <- varImp(model, scale = TRUE)$importance
#       importance_file <- paste0("new20250624_importance_", tools::file_path_sans_ext(file), "_", target_var, "_", today, ".csv")
#       write.csv(importance, importance_file)
#     }, silent = TRUE)

#     # 軸スケール決定関数
#     get_axis_limit <- function(values) {
#       max_val <- max(values, na.rm = TRUE)
#       if (max_val <= 1.5) {
#         return(ceiling(max_val / 0.1) * 0.1)
#       } else if (max_val <= 5) {
#         return(ceiling(max_val / 0.5) * 0.5)
#       } else if (max_val <= 30) {
#         return(ceiling(max_val / 2) * 2)
#       } else {
#         return(ceiling(max_val / 5) * 5)
#       }
#     }

#     range_max <- get_axis_limit(c(obs, pred))

#     # プロット
#     p <- ggplot(data.frame(Predicted = pred, Observed = obs), aes(x = Observed, y = Predicted)) +
#       geom_point(color = "blue", alpha = 0.7) +
#       geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") +
#       scale_x_continuous(limits = c(0, range_max)) +
#       scale_y_continuous(limits = c(0, range_max)) +
#       coord_fixed(ratio = 1) +
#       labs(
#         title = paste0(target_var, " Prediction (", file, ")"),
#         x = "Observed",
#         y = "Predicted"
#       ) +
#       theme_minimal() +
#       theme(
#         plot.title = element_text(hjust = 0.5, size = 14),
#         plot.subtitle = element_text(hjust = 0.5, size = 10)
#       ) +
#       annotate("text", x = range_max * 0.05, y = range_max * 0.95, hjust = 0, vjust = 1, size = 4,
#                label = paste0("RMSE = ", RMSE_val, "\nMAE = ", MAE_val, "\nRPD = ", RPD_val)) +
#       annotate("text", x = range_max * 0.95, y = range_max * 0.05, hjust = 1, vjust = 0, size = 5,
#                label = paste0("R² = ", R2))

#     # モデル保存
#     model_data_bundle <- list(
#       model = model,
#       training_data = df
#     )
#     rds_file <- paste0("new20250624_model_data_", tools::file_path_sans_ext(file), "_", target_var, "_", today, ".rds")
#     saveRDS(model_data_bundle, file = rds_file)

#     # PDF出力
#     plot_file <- paste0("new20250624_plot_", tools::file_path_sans_ext(file), "_", target_var, "_", today, ".pdf")
#     pdf(plot_file, width = 6, height = 6)
#     print(p)
#     dev.off()
#   }
# }

# #--- 結果の保存 ---#
# output_file <- paste0("new20250624_WM_summary_", today, ".csv")
# write.csv(result_matrix, output_file, row.names = TRUE)
# cat("\nSummary saved as", output_file, "\n")



=== Processing: n_base.csv ===
Final dataset size: 358 12 

---
Training model for: Jsc on n_base.csv 


Warning message:
"model fit failed for Fold01: num.labels=3, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold01: num.labels=5, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold01: num.labels=7, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold02: num.labels=3, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold02: num.labels=5, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold02: num.labels=7, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"


[1] "note: Some of your new data are out of the previously specified range"
[1] "note: Some of your new data are out of the previously specified range"
[1] "note: Some of your new data are out of the previously specified range"


Warning message:
"model fit failed for Fold03: num.labels=3, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold03: num.labels=5, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold03: num.labels=7, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold04: num.labels=3, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold04: num.labels=5, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold04: num.labels=7, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning me

[1] "note: Some of your new data are out of the previously specified range"
[1] "note: Some of your new data are out of the previously specified range"
[1] "note: Some of your new data are out of the previously specified range"


...


=== Processing: n_base_OH.csv ===
Final dataset size: 358 143 

---
Training model for: Jsc on n_base_OH.csv 


Warning message:
"model fit failed for Fold01: num.labels=3, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold01: num.labels=5, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold01: num.labels=7, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold01: num.labels=3, type.mf=TRIANGLE Error in if (data[h, i] <= varinp.mf[2, j]) temp <- 0 else if (data[h,  : 
  missing value where TRUE/FALSE needed
"
Warning message:
"model fit failed for Fold01: num.labels=5, type.mf=TRIANGLE Error in if (data[h, i] <= varinp.mf[2, j]) temp <- 0 else if (data[h,  : 
  missing value where TRUE/FALSE needed
"
Warning message:
"model fit failed for Fold01: num.labels=7, type.mf=TRIANGLE Error in if (data[h, i] <= varinp.mf[

Warning message:
"model fit failed for Fold08: num.labels=5, type.mf=TRIANGLE Error in if (data[h, i] <= varinp.mf[2, j]) temp <- 0 else if (data[h,  : 
  missing value where TRUE/FALSE needed
"
Warning message:
"model fit failed for Fold08: num.labels=7, type.mf=TRIANGLE Error in if (data[h, i] <= varinp.mf[2, j]) temp <- 0 else if (data[h,  : 
  missing value where TRUE/FALSE needed
"
Warning message:
"model fit failed for Fold09: num.labels=3, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold09: num.labels=5, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold09: num.labels=7, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold09: num.labels=3, type.mf=TRIANGLE Error in if (data[h, i] <= varinp.mf[

Something is wrong; all the RMSE metric values are missing:
      RMSE        Rsquared        MAE     
 Min.   : NA   Min.   : NA   Min.   : NA  
 1st Qu.: NA   1st Qu.: NA   1st Qu.: NA  
 Median : NA   Median : NA   Median : NA  
 Mean   :NaN   Mean   :NaN   Mean   :NaN  
 3rd Qu.: NA   3rd Qu.: NA   3rd Qu.: NA  
 Max.   : NA   Max.   : NA   Max.   : NA  
 NA's   :6     NA's   :6     NA's   :6    


ERROR: Error: Stopping


In [7]:
#--- 事前に必要なパッケージ ---#
library(caret)
library(Metrics)
library(ggplot2)
library(lattice)
library(frbs)  # WM用

#--- ファイルと日付の初期化 ---#
file_names <- c(
  "n_base.csv", "n_base_OH.csv", "n_base_FP.csv",
  "n_all.csv", "n_all_OH.csv", "n_all_FP.csv",
  "m_base.csv", "m_base_OH.csv", "m_base_FP.csv",
  "m_all.csv", "m_all_OH.csv", "m_all_FP.csv"
)
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/"
today <- format(Sys.Date(), "%Y%m%d")

target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
metric_names <- c("Best_num.labels", "Best_type.mf", "R2", "RMSE", "MAE", "RPD", "n_samples", "n_features")
result_matrix <- matrix(nrow = length(metric_names) * length(target_vars), ncol = length(file_names))
rownames(result_matrix) <- as.vector(t(outer(metric_names, target_vars, paste, sep = "_")))
colnames(result_matrix) <- file_names

#--- メインループ ---#
for (file in file_names) {
  filepath <- paste0(base_path, file)
  cat("\n=== Processing:", file, "===\n")
  df_all <- read.csv(filepath)
  cat("Final dataset size:", dim(df_all)[1], dim(df_all)[2], "\n")

  feature_vars <- setdiff(colnames(df_all), target_vars)

  for (target_var in target_vars) {
    cat("\n---\nTraining model for:", target_var, "on", file, "\n")

    df <- df_all[, c(feature_vars, target_var)]
    df <- df[complete.cases(df), ]
    df <- df[sapply(df, is.numeric)]  # 数値列のみ
    df <- predict(preProcess(df, method = "range"), df)

    #--- ハイパーパラメータ設定 ---#
    tune_grid <- expand.grid(
      num.labels = c(3, 5, 7),
      type.mf = c("GAUSS", "TRIANGLE")
    )

    ctrl <- trainControl(method = "cv", number = 10)

    model <- train(
      formula(paste(target_var, "~ .")),
      data = df,
      method = "WM",
      metric = "RMSE",
      trControl = ctrl,
      tuneGrid = tune_grid
    )

    pred <- predict(model, df)
    obs <- df[[target_var]]

    R2 <- round(cor(pred, obs)^2, 3)
    RMSE_val <- round(rmse(obs, pred), 3)
    MAE_val <- round(mae(obs, pred), 3)
    RPD_val <- round(sd(obs) / RMSE_val, 3)

    cat("Best params: num.labels =", model$bestTune$num.labels, ", type.mf =", model$bestTune$type.mf, "\n")
    cat(file, target_var, ": R2 =", R2, ", RMSE =", RMSE_val, ", MAE =", MAE_val, ", RPD =", RPD_val, "\n")

    # 結果保存
    result_matrix[paste0("Best_num.labels_", target_var), file] <- model$bestTune$num.labels
    result_matrix[paste0("Best_type.mf_", target_var), file]    <- model$bestTune$type.mf
    result_matrix[paste0("R2_", target_var), file]              <- R2
    result_matrix[paste0("RMSE_", target_var), file]           <- RMSE_val
    result_matrix[paste0("MAE_", target_var), file]            <- MAE_val
    result_matrix[paste0("RPD_", target_var), file]            <- RPD_val
    result_matrix[paste0("n_samples_", target_var), file]      <- nrow(df)
    result_matrix[paste0("n_features_", target_var), file]     <- ncol(df) - 1

    # 軸スケール決定関数
    get_axis_limit <- function(values) {
      max_val <- max(values, na.rm = TRUE)
      if (max_val <= 1.5) return(ceiling(max_val / 0.1) * 0.1)
      if (max_val <= 5)   return(ceiling(max_val / 0.5) * 0.5)
      if (max_val <= 30)  return(ceiling(max_val / 2) * 2)
      return(ceiling(max_val / 5) * 5)
    }

    range_max <- get_axis_limit(c(obs, pred))

    p <- ggplot(data.frame(Predicted = pred, Observed = obs), aes(x = Observed, y = Predicted)) +
      geom_point(color = "blue", alpha = 0.7) +
      geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") +
      scale_x_continuous(limits = c(0, range_max)) +
      scale_y_continuous(limits = c(0, range_max)) +
      coord_fixed(ratio = 1) +
      labs(
        title = paste0(target_var, " Prediction (", file, ")"),
        x = "Observed", y = "Predicted"
      ) +
      theme_minimal() +
      annotate("text", x = range_max * 0.05, y = range_max * 0.95, hjust = 0, vjust = 1, size = 4,
               label = paste0("RMSE = ", RMSE_val, "\nMAE = ", MAE_val, "\nRPD = ", RPD_val)) +
      annotate("text", x = range_max * 0.95, y = range_max * 0.05, hjust = 1, vjust = 0, size = 5,
               label = paste0("R² = ", R2))

    # モデル＋データ保存
    model_data_bundle <- list(model = model, training_data = df)
    rds_file <- paste0("new20250624_model_data_", tools::file_path_sans_ext(file), "_", target_var, "_WM_", today, ".rds")
    saveRDS(model_data_bundle, file = rds_file)

    # PDF出力
    plot_file <- paste0("new20250624_plot_", tools::file_path_sans_ext(file), "_", target_var, "_WM_", today, ".pdf")
    pdf(plot_file, width = 6, height = 6)
    print(p)
    dev.off()
  }
}

#--- 最終出力 ---#
output_file <- paste0("new20250624_WM_summary_", today, ".csv")
write.csv(result_matrix, output_file, row.names = TRUE)
cat("\nSummary saved as", output_file, "\n")



=== Processing: n_base.csv ===
Final dataset size: 358 12 

---
Training model for: Jsc on n_base.csv 


Warning message:
"model fit failed for Fold01: num.labels=3, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold01: num.labels=5, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold01: num.labels=7, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold02: num.labels=3, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold02: num.labels=5, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold02: num.labels=7, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"


[1] "note: Some of your new data are out of the previously specified range"
[1] "note: Some of your new data are out of the previously specified range"
[1] "note: Some of your new data are out of the previously specified range"
[1] "note: Some of your new data are out of the previously specified range"
[1] "note: Some of your new data are out of the previously specified range"
[1] "note: Some of your new data are out of the previously specified range"


Warning message:
"model fit failed for Fold03: num.labels=3, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold03: num.labels=5, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold03: num.labels=7, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold04: num.labels=3, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold04: num.labels=5, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold04: num.labels=7, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning me

[1] "note: Some of your new data are out of the previously specified range"
[1] "note: Some of your new data are out of the previously specified range"
[1] "note: Some of your new data are out of the previously specified range"


...


=== Processing: n_base_OH.csv ===
Final dataset size: 358 143 

---
Training model for: Jsc on n_base_OH.csv 


Warning message:
"model fit failed for Fold01: num.labels=3, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold01: num.labels=5, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold01: num.labels=7, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold01: num.labels=3, type.mf=TRIANGLE Error in if (data[h, i] <= varinp.mf[2, j]) temp <- 0 else if (data[h,  : 
  missing value where TRUE/FALSE needed
"
Warning message:
"model fit failed for Fold01: num.labels=5, type.mf=TRIANGLE Error in if (data[h, i] <= varinp.mf[2, j]) temp <- 0 else if (data[h,  : 
  missing value where TRUE/FALSE needed
"
Warning message:
"model fit failed for Fold01: num.labels=7, type.mf=TRIANGLE Error in if (data[h, i] <= varinp.mf[

Warning message:
"model fit failed for Fold08: num.labels=5, type.mf=TRIANGLE Error in if (data[h, i] <= varinp.mf[2, j]) temp <- 0 else if (data[h,  : 
  missing value where TRUE/FALSE needed
"
Warning message:
"model fit failed for Fold08: num.labels=7, type.mf=TRIANGLE Error in if (data[h, i] <= varinp.mf[2, j]) temp <- 0 else if (data[h,  : 
  missing value where TRUE/FALSE needed
"
Warning message:
"model fit failed for Fold09: num.labels=3, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold09: num.labels=5, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold09: num.labels=7, type.mf=GAUSS Error in fuzzifier(data, num.varinput, num.labels, var.mf) : 
  object 'temp' not found
"
Warning message:
"model fit failed for Fold09: num.labels=3, type.mf=TRIANGLE Error in if (data[h, i] <= varinp.mf[

Something is wrong; all the RMSE metric values are missing:
      RMSE        Rsquared        MAE     
 Min.   : NA   Min.   : NA   Min.   : NA  
 1st Qu.: NA   1st Qu.: NA   1st Qu.: NA  
 Median : NA   Median : NA   Median : NA  
 Mean   :NaN   Mean   :NaN   Mean   :NaN  
 3rd Qu.: NA   3rd Qu.: NA   3rd Qu.: NA  
 Max.   : NA   Max.   : NA   Max.   : NA  
 NA's   :6     NA's   :6     NA's   :6    


ERROR: Error: Stopping


In [2]:
install.packages("frbs")


Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done

